In [0]:
SHOW CATALOGS; 

In [0]:
SELECT current_catalog(), current_schema(); -- Permite obtener los catálogos y esquemas por defecto dentro del Workspace. 

In [0]:
SHOW SCHEMAS;

In [0]:
SHOW SCHEMAS IN kissflow;

In [0]:
SHOW TABLES;

In [0]:
SELECT *
FROM prodsqldwh.adls.nav_customer_dim
LIMIT 100;

In [0]:
USE CATALOG prodsqldwh;
USE SCHEMA adls;

SELECT current_catalog(), current_schema();


In [0]:
SHOW TABLES;

In [0]:
SELECT * 
FROM prodsqldwh.adls.epuma_customer_ui
LIMIT 100;

In [0]:
SHOW VOLUMES;

In [0]:
-- POR MEDIO DEL STATEMENT LIST E INDICANDO EL PATH DE UN VOLUMEN DENTRO DE DATABRICKS, SE PUEDEN VER TODOS LOS ELEMENTOS DENTRO DE DICHO VOLUMEN.
LIST '/Volumes/prodsqldwh/adls/navfiles'  

In [0]:
-- TAMBIÉN PODEMOS VISUALIZAR LOS ARCHIVOS DENTRO DE UN VOLUMEN, ESPECIFICANDO LA RUTA COMPLETA DE DICHO ARCHIVO DENTRO DE DATABRICKS, ANTECEDIDO DEL FORMATO EN EL QUE QUEREMOS VISUALIZARLO.
SELECT * 
FROM TEXT.`/Volumes/prodsqldwh/adls/navfiles`

In [0]:
SELECT * 
FROM CSV.`/Volumes/prodsqldwh/adls/navfiles`


In [0]:
-- READ_FILES FUNCTION:
CREATE OR REPLACE TABLE epuma_customer_readfiles_func AS
SELECT * 
FROM read_files(
  '/Volumes/prodsqldwh/adls/navfiles/nav_customer_dim.csv', 
  FORMAT => 'CSV',
  HEADER => TRUE
);

In [0]:
SHOW TABLES;

In [0]:
SELECT * 
FROM epuma_customer_readfiles_func;

In [0]:
-- COPY INTO STATEMENT:
DROP TABLE IF EXISTS epuma_customer_copyinto;

CREATE TABLE epuma_customer_copyinto;

COPY INTO epuma_customer_copyinto
FROM '/Volumes/prodsqldwh/adls/navfiles/nav_customer_dim.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true', 
  'inferSchema' = 'false',
  'mergeSchema' = 'true'
  )
COPY_OPTIONS ('mergeSchema' = 'true')


In [0]:
COPY INTO epuma_customer_copyinto
FROM '/Volumes/prodsqldwh/adls/navfiles/nav_customer_dim.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true', 
  'inferSchema' = 'false', 
  'mergeSchema' = 'true'
)
COPY_OPTIONS ('mergeSchema' = 'true')

-- En la ejecución previa, la información ya había sido copiada dentro de la tabla epuma_customer_copyinto, 
-- por lo cual si se vuelve a copiar el mismo archivo nav_customer_dim.csv dentro de la tabla, el sistema reconoce que los registros ya habían sido previamente copiados y no los agrega a la Delta Table, por lo cual no se duplica la info. al utilizar COPY INTO.


In [0]:
SHOW TABLES; 

In [0]:
SELECT * 
FROM epuma_customer_copyinto;

In [0]:
DROP TABLE IF EXISTS epuma_cust_bronze_raw;

CREATE TABLE epuma_cust_bronze_raw (
  customer_id INT,
  customer_name VARCHAR(30),
  customer_address VARCHAR(30)
);

INSERT INTO epuma_cust_bronze_raw (
  customer_id, 
  customer_name, 
  customer_address
)
VALUES 
  (125698, 'John', 'Calle 5'), 
  (200678, 'Carlos', 'Cr 58-9'),
  (569989, 'Jose', 'Av. 42-7');

In [0]:
CREATE OR REFRESH STREAMING TABLE epuma_cust_silver_streaming
--SCHEDULE EVERY 1 HOUR
SELECT *, 
        current_timestamp() AS IngestDateTime
FROM STREAM epuma_cust_bronze_raw;

In [0]:
DESCRIBE HISTORY epuma_cust_silver_streaming;


In [0]:
SELECT * 
FROM epuma_cust_silver_streaming;


In [0]:
INSERT INTO epuma_cust_bronze_raw (
  customer_id, 
  customer_name, 
  customer_address
)
VALUES 
(005695, 'Juan', 'Calle 10'), 
(468595, 'Will', 'Calle 15-5'), 
(785469, 'Pablo', 'Cr 45-78 5');

In [0]:
REFRESH STREAMING TABLE epuma_cust_silver_streaming;

In [0]:
DESCRIBE HISTORY epuma_cust_silver_streaming;

In [0]:
SELECT * 
FROM epuma_cust_silver_streaming; 

<b> Recap: Statements and functions for data exploration and ingestion: </b>

  - SHOW CATALOG;
  - SHOW SCHEMA;
  - SHOW TABLES; 
  - SHOW VOLUMES;
  - USE CATALOG CatalogName;
  - USE SCHEMA SchemaName;
  - SELECT current_catalog(), current_schema();
  - CREATE OR REPLACE TABLE TableName;
  - DROP TABLE IF EXISTS TableName;
  - INSERT INTO TableName (ColumnsNames & ColumnsTypes)
  - VALUES (value1), (value2);
  - read_files(path, format => '', header => '')
  - COPY INTO / FROM / FILEFORMAT / FORMAT_OPTIONS / COPY_OPTIONS
  - CREATE OR REFRESH STREAMING TABLE TableName;
  - SCHEDULE EVERY 1 HOUR
  - FROM STREAM TableName;
  - DESCRIBE HISTORY TableName; 